# 07 - Resultados no Looker e insights do P2 (Fase 2 · ML)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho (Item2Vec)**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`

---

### Objetivo
Card da task: *"Criar os insights no Looker e estruturar os resultados adquiridos durante a comparação dos modelos e a evolução de baselines. Para modelos não supervisionados, deve ter a aplicação de testes estatísticos e avaliação dos resultados."*

Este notebook não treina nada novo. Ele reúne o que os notebooks 05 (comparação) e 06 (evolução) produziram, acrescenta a análise não supervisionada que faltava (**que "missões de compra" o Item2Vec aprendeu e por quê**) e grava tudo em tabelas prontas para o Looker.

### Por que analisar o Item2Vec como modelo não supervisionado
O Item2Vec não tem rótulo: ele aprende um vetor por SKU só olhando quais produtos aparecem juntos. O resultado final é "recomende estes 5". Seguindo a orientação da weekly de 26/11, quem explica **por que** são esses 5 somos nós:
1. agrupar os vetores e descrever cada grupo com estatística (categoria dominante, preço, popularidade);
2. testar se os grupos e os vizinhos fazem sentido ou se poderiam ter saído por acaso.

### Entradas e saídas
| Lê | Grava (Delta na Gold da Fase 2 **e** Azure SQL Server, schema `squad2`) |
|---|---|
| `item2vec_skus`, `item2vec_cestas_treino`, `item2vec_holdout`, `item2vec_vetores` (v2), `item2vec_recomendacoes_holdout` (v1 e v2), `p2_comparacao_recomendacoes` (notebook 05), Silver `ecommerce_produtos` (preço) | `ml_p2_comparacao_modelos`, `ml_p2_evolucao_item2vec`, `ml_p2_missoes_compra`, `ml_p2_missoes_skus`, `ml_p2_testes_estatisticos`, `ml_p2_insights` |

O Looker da squad lê o SQL Server (mesmo caminho da Gold da Fase 1). O passo a passo da página está em `squad2/dashboard/GUIA_LOOKER_P2_ML.md`.

In [ ]:
%pip install scikit-learn==1.5.2 python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, caminhos e parâmetros

In [ ]:
import os
import math
from collections import Counter
from datetime import datetime
from itertools import combinations
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.stats import chi2_contingency, kruskal

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_user = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"
assert sql_host and sql_database and sql_user and sql_password, "Credenciais do Azure SQL Server não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
opcoes_sql = {"host": sql_host, "port": "1433", "user": sql_user, "password": sql_password, "database": sql_database}

RAIZ = f"abfss://squad2@{fqdn}"
GOLD = f"{RAIZ}/fase2_ml/gold"
ENTRADAS = {
    "skus": f"{GOLD}/item2vec_skus",
    "cestas_treino": f"{GOLD}/item2vec_cestas_treino",
    "holdout": f"{GOLD}/item2vec_holdout",
    "vetores": f"{GOLD}/item2vec_vetores",
    "recs_item2vec": f"{GOLD}/item2vec_recomendacoes_holdout",
    "recs_comparacao": f"{GOLD}/p2_comparacao_recomendacoes",
    "produtos": f"{RAIZ}/silver/ecommerce_produtos",
}
SAIDAS = ["ml_p2_comparacao_modelos", "ml_p2_evolucao_item2vec", "ml_p2_missoes_compra",
          "ml_p2_missoes_skus", "ml_p2_testes_estatisticos", "ml_p2_insights"]

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def gravar_delta(df, destino, modo="overwrite"):
    w = df.write.format("delta").options(**adls_options).mode(modo)
    (w.option("overwriteSchema", "true") if modo == "overwrite" else w.option("mergeSchema", "true")).save(destino)

print(f"scikit-learn {sklearn.__version__}")

In [ ]:
SEED = 42
TOP_K = 5
K_CANDIDATOS = list(range(4, 21))   # números de grupos testados no K-Means
N_ORDENS_KMEANS = 10               # ordens diferentes das linhas para escolher K e medir a estabilidade
N_PERMUTACOES = 1000                # testes de permutação (coerência e Cramér's V)
N_BOOTSTRAP = 2000                  # IC da diferença de Hit@5 entre modelos
N_COERENCIA = 200                   # SKUs mais vendidos usados na coerência (mesma medida do notebook 06)
ALFA = 0.05
rng = np.random.default_rng(SEED)

# FP-Growth v1 do Lucas: não há recomendação por cesta gravada, só o resultado que ele mediu
# nas mesmas 336 cestas em 13/11/2026 (notebook run_pipeline_ml, min_support 0,003, sem período).
FP_V1_LUCAS = {"acertos": 11, "cestas": 336, "cobertura_catalogo": 0.0922, "min_support": 0.003}
MIN_SUPPORT_V2_LUCAS = 0.01

## 1. Leitura
Tudo vai para pandas: são ~1,2 mil cestas de treino, 336 de teste e 3,8 mil SKUs.

In [ ]:
skus = ler_delta(ENTRADAS["skus"]).toPandas()
treino = ler_delta(ENTRADAS["cestas_treino"]).orderBy("id_pedido").toPandas()
holdout = ler_delta(ENTRADAS["holdout"]).orderBy("id_pedido").toPandas()
vetores = ler_delta(ENTRADAS["vetores"]).toPandas()
recs_item2vec = ler_delta(ENTRADAS["recs_item2vec"]).toPandas()
recs_comparacao = ler_delta(ENTRADAS["recs_comparacao"]).toPandas()
produtos = ler_delta(ENTRADAS["produtos"]).select("sku", "preco_lista").toPandas()

In [ ]:
cestas_treino = [list(s) for s in treino["skus"]]
subcategoria_de = dict(zip(skus["sku"], skus["subcategoria"]))
raiz_de = dict(zip(skus["sku"], skus["categoria_raiz"]))
preco_de = dict(zip(produtos["sku"], produtos["preco_lista"]))
N_CATALOGO = len(skus)

conjuntos_treino = [frozenset(c) for c in cestas_treino]
holdout["repete_treino"] = [any(frozenset([*c, r]) <= s for s in conjuntos_treino)
                            for c, r in zip(holdout["carrinho"], holdout["item_removido"])]
repete = holdout.set_index("id_pedido")["repete_treino"]
copias = pd.Series(conjuntos_treino).value_counts()
CESTAS_COPIADAS = int(copias[copias > 1].sum())

print(f"Treino: {len(cestas_treino):,} cestas ({CESTAS_COPIADAS} são cópias exatas de outra) | "
      f"hold-out: {len(holdout)} cestas, {int(repete.sum())} repetem o treino | "
      f"vetores v2: {len(vetores):,} SKUs | catálogo: {N_CATALOGO:,} SKUs")

## 2. Tabela única da comparação (`ml_p2_comparacao_modelos`)
É a tabela pedida pelo tech lead: os mesmos 336 pedidos de teste para todos os modelos. Cada linha é recalculada a partir das recomendações gravadas cesta a cesta, e não copiada dos notebooks anteriores:
* **Top-5 mais vendidos, FP-Growth v2 e FP-Growth calibrado:** recomendações do notebook 05 (`p2_comparacao_recomendacoes`);
* **Item2Vec v1 e v2:** recomendações dos notebooks 03 e 06 (`item2vec_recomendacoes_holdout`);
* **FP-Growth v1 (Lucas):** só existe o resultado agregado que ele mediu nas mesmas 336 cestas, por isso as colunas que dependem de cada cesta ficam vazias;
* **FP-Growth v2 sem período:** com `min_support` 0,01, nenhum par de SKUs chega ao mínimo de ocorrências nem no treino inteiro (conferido abaixo). Sem regra de SKU, a recomendação sai só da camada de categoria e da popularidade, que é igual com ou sem período. As recomendações são, portanto, as mesmas da v2 com período.

Os testes estatísticos comparam cada modelo com o Item2Vec v2:
* **McNemar exato:** olha só as cestas em que um acertou e o outro errou.
* **Bootstrap pareado (2.000 reamostragens das 336 cestas):** dá o intervalo de 95% da diferença de Hit@5. Se o intervalo cruza o zero, a diferença pode ser ruído.

In [ ]:
pares = Counter(p for c in cestas_treino for p in combinations(sorted(set(c)), 2))
MAX_PAR = max(pares.values())
MIN_OCORRENCIAS_V2 = math.ceil(MIN_SUPPORT_V2_LUCAS * len(cestas_treino))
print(f"FP-Growth v2 sem período: min_support {MIN_SUPPORT_V2_LUCAS} pede {MIN_OCORRENCIAS_V2} ocorrências; "
      f"o par mais frequente aparece em {MAX_PAR} cestas -> {'nenhuma regra de SKU' if MAX_PAR < MIN_OCORRENCIAS_V2 else 'há regras'}")


def wilson(acertos, n, z=1.959963984540054):
    p = acertos / n
    d = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / d
    margem = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, centro - margem), min(1.0, centro + margem)


def mcnemar_exato(a, b):
    so_a, so_b = int(np.sum(a & ~b)), int(np.sum(~a & b))
    n = so_a + so_b
    if n == 0:
        return so_a, so_b, 1.0
    return so_a, so_b, min(1.0, 2 * sum(math.comb(n, i) for i in range(min(so_a, so_b) + 1)) / 2 ** n)


def bootstrap_diferenca(a, b, n=N_BOOTSTRAP):
    """IC 95% de mean(a) - mean(b), reamostrando as mesmas cestas para os dois modelos."""
    idx = rng.integers(0, len(a), size=(n, len(a)))
    difs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return float(np.percentile(difs, 2.5)), float(np.percentile(difs, 97.5))


def por_cesta(df):
    """Uma linha por cesta de teste, na ordem do hold-out, com acerto e acerto de subcategoria."""
    df = df.set_index("id_pedido").loc[holdout["id_pedido"]]
    recs = [list(r) for r in df["recomendacoes"]]
    acerto = np.array([r in rs for r, rs in zip(df["item_removido"], recs)])
    sub = np.array([subcategoria_de.get(r) in {subcategoria_de.get(x) for x in rs} for r, rs in zip(df["item_removido"], recs)])
    return recs, acerto, sub


FONTES = {
    "top5_mais_vendidos": ("Top-5 mais vendidos", "Baseline", "notebook 05", recs_comparacao[recs_comparacao["modelo"] == "top5_mais_vendidos"]),
    "fpgrowth_v2_com_periodo": ("FP-Growth v2 com período", "Regras de associação", "notebook 05 (lógica da v2 do Lucas)", recs_comparacao[recs_comparacao["modelo"] == "fpgrowth_v2_lucas"]),
    "fpgrowth_v2_sem_periodo": ("FP-Growth v2 sem período", "Regras de associação", "notebook 05 (sem regra de SKU, igual à v2 com período)", recs_comparacao[recs_comparacao["modelo"] == "fpgrowth_v2_lucas"]),
    "fpgrowth_calibrado": ("FP-Growth calibrado (piso de 2)", "Regras de associação", "notebook 05", recs_comparacao[recs_comparacao["modelo"] == "fpgrowth_calibrado"]),
    "item2vec_v1": ("Item2Vec v1", "Embeddings", "notebook 03", recs_item2vec[(recs_item2vec["modelo"] == "item2vec_com_fallback") & (recs_item2vec["versao"] == "v1_baseline")]),
    "item2vec_v2": ("Item2Vec v2", "Embeddings", "notebook 06", recs_item2vec[(recs_item2vec["modelo"] == "item2vec_com_fallback") & (recs_item2vec["versao"] == "v2")]),
}
cestas = {nome: por_cesta(df) for nome, (_, _, _, df) in FONTES.items()}
rep_mask = holdout["repete_treino"].to_numpy()
ref = cestas["item2vec_v2"][1]

linhas = []
for ordem, nome in enumerate(["top5_mais_vendidos", "fp_v1", "fpgrowth_v2_com_periodo", "fpgrowth_v2_sem_periodo",
                              "fpgrowth_calibrado", "item2vec_v1", "item2vec_v2"], start=1):
    if nome == "fp_v1":
        k, n = FP_V1_LUCAS["acertos"], FP_V1_LUCAS["cestas"]
        inf, sup = wilson(k, n)
        linhas.append({"ordem": ordem, "modelo": "FP-Growth v1 (anterior)", "familia": "Regras de associação",
                       "fonte": "Lucas, run_pipeline_ml (13/11/2026)", "cestas": n, "acertos": k, "hit_at_5": k / n,
                       "ic95_inf": inf, "ic95_sup": sup, "precision_at_5": k / n / TOP_K,
                       "cobertura_catalogo": FP_V1_LUCAS["cobertura_catalogo"]})
        continue
    rotulo, familia, fonte, _ = FONTES[nome]
    recs, acerto, sub = cestas[nome]
    k, n = int(acerto.sum()), len(acerto)
    inf, sup = wilson(k, n)
    linha = {"ordem": ordem, "modelo": rotulo, "familia": familia, "fonte": fonte, "cestas": n, "acertos": k,
             "hit_at_5": k / n, "ic95_inf": inf, "ic95_sup": sup, "precision_at_5": k / n / TOP_K,
             "cobertura_catalogo": len({x for r in recs for x in r}) / N_CATALOGO,
             "skus_recomendados": len({x for r in recs for x in r}),
             "acertos_cestas_repetidas": int(acerto[rep_mask].sum()), "acertos_cestas_novas": int(acerto[~rep_mask].sum()),
             "hit_at_5_cestas_novas": float(acerto[~rep_mask].mean()), "acerto_subcategoria": float(sub.mean())}
    if nome != "item2vec_v2":
        so_v2, so_modelo, p = mcnemar_exato(ref, acerto)
        d_inf, d_sup = bootstrap_diferenca(ref.astype(float), acerto.astype(float))
        linha.update({"dif_hit_item2vec_v2_menos_modelo": float(ref.mean() - acerto.mean()),
                      "dif_ic95_inf": d_inf, "dif_ic95_sup": d_sup, "mcnemar_p_vs_item2vec_v2": p,
                      "conclusao_vs_item2vec_v2": "Item2Vec v2 melhor" if p < ALFA and ref.sum() > k else
                                                  ("modelo melhor" if p < ALFA else "empate estatístico")})
    linhas.append(linha)

comparacao = pd.DataFrame(linhas)
contagens = ["skus_recomendados", "acertos_cestas_repetidas", "acertos_cestas_novas"]
comparacao[contagens] = comparacao[contagens].astype("Int64")   # inteiro com vazio (linha do FP-Growth v1)
exibir = comparacao.copy()
for c in ["hit_at_5", "ic95_inf", "ic95_sup", "cobertura_catalogo", "hit_at_5_cestas_novas", "acerto_subcategoria",
          "dif_hit_item2vec_v2_menos_modelo", "dif_ic95_inf", "dif_ic95_sup"]:
    exibir[c] = (100 * exibir[c]).round(2)
display(exibir[["ordem", "modelo", "acertos", "hit_at_5", "ic95_inf", "ic95_sup", "cobertura_catalogo",
                "acertos_cestas_repetidas", "acertos_cestas_novas", "dif_hit_item2vec_v2_menos_modelo",
                "dif_ic95_inf", "dif_ic95_sup", "mcnemar_p_vs_item2vec_v2", "conclusao_vs_item2vec_v2"]])

## 3. Evolução do Item2Vec (`ml_p2_evolucao_item2vec`)
A busca da v2 (notebook 06) treina dezenas de modelos e leva ~10 minutos. Em vez de repeti-la, esta tabela registra os números da **execução confirmada no Databricks em 23/11/2026** (validação dentro do treino com média de 3 seeds; hold-out com seed 42 e com 10 seeds). As linhas de hold-out com seed 42 são conferidas abaixo contra as recomendações gravadas.

In [ ]:
EVOLUCAO_NB06 = [
    # etapa, descricao, onde, hit, coerencia, cobertura
    (1, "v1 (notebook 03)", "validação (3 seeds)", 0.1093, 0.4163, 0.1352),
    (2, "Etapa 1: 200 épocas e ns_exponent 1,0", "validação (3 seeds)", 0.1987, 0.3533, 0.1597),
    (3, "Etapa 2: negative 15", "validação (3 seeds)", 0.2000, 0.3557, 0.1549),
    (4, "Etapa 3: subcategoria como token (= v2)", "validação (3 seeds)", 0.2013, 0.3807, 0.1545),
    (5, "v1", "hold-out (seed 42)", 0.0417, 0.4130, 0.1784),
    (6, "v2", "hold-out (seed 42)", 0.0655, 0.4150, 0.1869),
    (7, "v1", "hold-out (média de 10 seeds)", 0.0440, 0.4193, 0.1762),
    (8, "v2", "hold-out (média de 10 seeds)", 0.0631, 0.4011, 0.1877),
]
evolucao = pd.DataFrame(EVOLUCAO_NB06, columns=["ordem", "versao_ou_etapa", "onde_medido", "hit_at_5",
                                                "coerencia_vizinhos", "cobertura_catalogo"])
evolucao["fonte"] = "notebook 06 (Databricks, 23/11/2026)"

for nome, ordem in [("item2vec_v1", 5), ("item2vec_v2", 6)]:
    gravado = comparacao.loc[comparacao["modelo"] == FONTES[nome][0], "hit_at_5"].iloc[0]
    tabelado = evolucao.loc[evolucao["ordem"] == ordem, "hit_at_5"].iloc[0]
    assert abs(gravado - tabelado) < 0.0005, f"{nome}: Hit@5 gravado {gravado:.4f} difere do notebook 06 {tabelado:.4f}"
print("Hold-out com seed 42 confere com as recomendações gravadas.")
display(evolucao)

## 4. Missões de compra: o que o Item2Vec aprendeu (`ml_p2_missoes_*`)
Os vetores da v2 são agrupados com K-Means. Antes, cada vetor é normalizado: assim a distância usada no agrupamento equivale ao cosseno, a mesma medida que o modelo usa para recomendar.
* **Quantos grupos:** o K-Means depende da ordem em que os SKUs entram (a primeira execução no Databricks deu 11 grupos e a local deu 13, com os mesmos vetores). Por isso cada K, de 4 a 20, roda em 10 ordens diferentes das linhas, e o escolhido é o de maior **silhouette médio**. O silhouette vai de −1 a 1 e mede quanto cada SKU está mais perto do próprio grupo do que do vizinho.
* **Estabilidade:** o **ARI** (Adjusted Rand Index) compara os grupos de duas rodadas: 1 = mesmos grupos, 0 = concordância de acaso. A média entre os 45 pares de rodadas vai para a tabela de testes.
* **Agrupamento final:** os vetores ordenados por SKU, com seed fixa, para o resultado ser o mesmo em qualquer máquina.
* **Nome de cada grupo:** a categoria raiz dominante e as subcategorias mais frequentes. Quando nenhuma categoria raiz passa de 60%, o grupo é chamado de "Mista" (são cestas de "compra do mês", que misturam corredores).
* **Estatísticas por grupo:** tamanho, categorias, preço (média, mediana, mínimo, máximo, desvio padrão), popularidade no treino e % das cestas de treino que contêm algum SKU do grupo.

In [ ]:
vetores = vetores.sort_values("sku").reset_index(drop=True)   # ordem fixa: o K-Means depende dela
X = np.vstack([np.asarray(v, dtype=float) for v in vetores["vetor"]])
X = X / np.linalg.norm(X, axis=1, keepdims=True)
info = pd.DataFrame({"sku": vetores["sku"]})
info["subcategoria"] = info["sku"].map(subcategoria_de)
info["categoria_raiz"] = info["sku"].map(raiz_de)
info["preco_lista"] = info["sku"].map(preco_de)
info["freq_treino"] = info["sku"].map(dict(zip(skus["sku"], skus["freq_treino"])))

rng_ordens = np.random.default_rng(SEED)   # separado do rng dos testes, para não mudar os sorteios deles
ordens = [rng_ordens.permutation(len(X)) for _ in range(N_ORDENS_KMEANS)]

def kmeans_na_ordem(k, ordem):
    rotulos = np.empty(len(X), dtype=int)
    rotulos[ordem] = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X[ordem]).labels_
    return rotulos

silhuetas = []
for k in K_CANDIDATOS:
    rodadas = [kmeans_na_ordem(k, o) for o in ordens]
    sil = [silhouette_score(X, r, metric="cosine") for r in rodadas]
    ari = [adjusted_rand_score(a, b) for a, b in combinations(rodadas, 2)]
    silhuetas.append({"k": k, "silhouette_medio": np.mean(sil), "silhouette_dp": np.std(sil),
                      "ari_medio": np.mean(ari), "ari_minimo": np.min(ari)})
silhuetas = pd.DataFrame(silhuetas)
escolhido = silhuetas.sort_values(["silhouette_medio", "k"], ascending=[False, True]).iloc[0]
K = int(escolhido["k"])
ARI_MEDIO, ARI_MINIMO = float(escolhido["ari_medio"]), float(escolhido["ari_minimo"])
display(silhuetas.round(4))

km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
SILHOUETTE = float(silhouette_score(X, km.labels_, metric="cosine"))
tamanhos = pd.Series(km.labels_).value_counts()
renumera = {g: i for i, g in enumerate(tamanhos.index, start=1)}   # grupo 1 = o maior
info["grupo"] = [renumera[g] for g in km.labels_]
print(f"K-Means com {K} grupos (silhouette {SILHOUETTE:.3f}; ARI médio entre ordens {ARI_MEDIO:.2f})")

In [ ]:
def nome_missao(g, d):
    """G01 Pets: Ração Cães, Ração Gatos | G06 Mista Higiene & Cuidados Pessoais + Bebidas: Refrigerantes, Azeite"""
    raizes = d["categoria_raiz"].value_counts(normalize=True)
    subs = ", ".join(d["subcategoria"].value_counts().index[:2])
    rotulo = raizes.index[0] if raizes.iloc[0] >= 0.6 else f"Mista {raizes.index[0]} + {raizes.index[1]}"
    return f"G{g:02d} {rotulo}: {subs}"

grupo_de = dict(zip(info["sku"], info["grupo"]))
cestas_por_grupo = Counter(g for c in cestas_treino for g in {grupo_de[s] for s in c if s in grupo_de})
preco_mediano_geral = float(info["preco_lista"].median())

missoes = []
for g, d in info.groupby("grupo"):
    raizes = d["categoria_raiz"].value_counts(normalize=True)
    missoes.append({
        "grupo": int(g), "missao": nome_missao(g, d), "n_skus": len(d), "pct_skus_vocabulario": len(d) / len(info),
        "categoria_raiz_dominante": raizes.index[0], "pct_categoria_dominante": float(raizes.iloc[0]),
        "segunda_categoria_raiz": raizes.index[1] if len(raizes) > 1 else None,
        "pct_segunda_categoria": float(raizes.iloc[1]) if len(raizes) > 1 else 0.0,
        "top_subcategorias": " · ".join(d["subcategoria"].value_counts().index[:3]),
        "preco_medio": float(d["preco_lista"].mean()), "preco_mediano": float(d["preco_lista"].median()),
        "preco_minimo": float(d["preco_lista"].min()), "preco_maximo": float(d["preco_lista"].max()),
        "preco_desvio_padrao": float(d["preco_lista"].std()),
        "preco_mediano_vs_geral_pct": float(d["preco_lista"].median() / preco_mediano_geral - 1),
        "freq_treino_media": float(d["freq_treino"].mean()), "freq_treino_mediana": float(d["freq_treino"].median()),
        "pct_cestas_treino_com_grupo": cestas_por_grupo[g] / len(cestas_treino),
    })
missoes = pd.DataFrame(missoes)
info["missao"] = info["grupo"].map(dict(zip(missoes["grupo"], missoes["missao"])))
display(missoes[["grupo", "missao", "n_skus", "pct_categoria_dominante", "top_subcategorias", "preco_mediano",
                 "preco_mediano_vs_geral_pct", "freq_treino_mediana", "pct_cestas_treino_com_grupo"]].round(3))

## 5. Testes estatísticos (`ml_p2_testes_estatisticos`)
| Teste | Pergunta | H0 (o que seria "acaso") |
|---|---|---|
| **Qui-quadrado + Cramér's V** (grupo × categoria raiz) | Os grupos do modelo correspondem a corredores do mercado? | grupo e categoria raiz são independentes |
| **Permutação do Cramér's V** | O V observado é maior do que daria com as categorias embaralhadas entre os SKUs? | V igual ao de rótulos aleatórios |
| **Kruskal-Wallis** (preço por grupo) | As missões têm faixas de preço diferentes? | a distribuição de preço é a mesma em todos os grupos |
| **Kruskal-Wallis** (popularidade por grupo) | Há missões de produtos mais e menos vendidos? | a distribuição de popularidade é a mesma |
| **Permutação da coerência dos vizinhos** | Os 5 vizinhos de cada SKU caem na mesma categoria raiz mais do que por acaso? | a coerência é a de categorias sorteadas |
| **Silhouette** | Os grupos são bem separados? | (descritivo: perto de 0 = grupos que se sobrepõem) |
| **Estabilidade (ARI entre ordens)** | Os grupos se repetem quando a ordem dos SKUs muda? | (descritivo: ≥ 0,8 estável; 0,6 a 0,8 moderado; < 0,6 instável) |
| **McNemar + bootstrap** (da seção 2) | O Item2Vec v2 acerta mais que o FP-Growth calibrado? | os dois acertam a mesma proporção |

Kruskal-Wallis é usado em vez de ANOVA porque preço e popularidade são assimétricos (poucos produtos caros ou muito vendidos).

In [ ]:
def cramer_v(rotulos_grupo, rotulos_categoria):
    tabela = pd.crosstab(rotulos_grupo, rotulos_categoria)
    chi2, p, gl, _ = chi2_contingency(tabela)
    return chi2, p, gl, float(np.sqrt(chi2 / (tabela.values.sum() * (min(tabela.shape) - 1))))

chi2, p_chi2, gl, V = cramer_v(info["grupo"], info["categoria_raiz"])
v_nulo = np.array([cramer_v(info["grupo"], rng.permutation(info["categoria_raiz"].to_numpy()))[3]
                   for _ in range(N_PERMUTACOES)])
p_v = (np.sum(v_nulo >= V) + 1) / (N_PERMUTACOES + 1)

h_preco, p_preco = kruskal(*[d["preco_lista"].to_numpy() for _, d in info.groupby("grupo")])
h_freq, p_freq = kruskal(*[d["freq_treino"].to_numpy() for _, d in info.groupby("grupo")])

# coerência: mesma medida do notebook 06 (5 vizinhos dos 200 SKUs mais vendidos)
indice = {s: i for i, s in enumerate(info["sku"])}
similares = X @ X.T
np.fill_diagonal(similares, -np.inf)
alvo = info.sort_values(["freq_treino", "sku"], ascending=[False, True])["sku"].head(N_COERENCIA).map(indice).to_numpy()
vizinhos = np.argsort(-similares[alvo], axis=1)[:, :5]
raizes = info["categoria_raiz"].to_numpy()

def coerencia(rotulos):
    return float((rotulos[vizinhos] == rotulos[alvo][:, None]).mean())

coer = coerencia(raizes)
coer_nula = np.array([coerencia(rng.permutation(raizes)) for _ in range(N_PERMUTACOES)])
p_coer = (np.sum(coer_nula >= coer) + 1) / (N_PERMUTACOES + 1)

linha_fp = comparacao[comparacao["modelo"] == FONTES["fpgrowth_calibrado"][0]].iloc[0]

def conclusao(p, sim, nao):
    return sim if p < ALFA else nao

testes = pd.DataFrame([
    {"teste": "Qui-quadrado (grupo x categoria raiz)", "objeto": "missões do Item2Vec v2",
     "estatistica": chi2, "valor_referencia": float(gl), "referencia": "graus de liberdade", "p_valor": p_chi2,
     "conclusao": conclusao(p_chi2, "grupos associados às categorias raiz", "sem associação")},
    {"teste": "Cramér's V (permutação)", "objeto": "missões do Item2Vec v2",
     "estatistica": V, "valor_referencia": float(v_nulo.mean()), "referencia": "V médio com categorias embaralhadas",
     "p_valor": p_v, "conclusao": conclusao(p_v, f"associação forte: V {V:.2f} contra {v_nulo.mean():.2f} por acaso", "não difere do acaso")},
    {"teste": "Kruskal-Wallis (preço por grupo)", "objeto": "missões do Item2Vec v2",
     "estatistica": float(h_preco), "valor_referencia": float(K - 1), "referencia": "graus de liberdade", "p_valor": float(p_preco),
     "conclusao": conclusao(p_preco, "as missões têm faixas de preço diferentes", "preço igual entre missões")},
    {"teste": "Kruskal-Wallis (popularidade por grupo)", "objeto": "missões do Item2Vec v2",
     "estatistica": float(h_freq), "valor_referencia": float(K - 1), "referencia": "graus de liberdade", "p_valor": float(p_freq),
     "conclusao": conclusao(p_freq, "há missões de produtos mais e menos vendidos", "popularidade igual entre missões")},
    {"teste": "Coerência dos vizinhos (permutação)", "objeto": "vetores do Item2Vec v2",
     "estatistica": coer, "valor_referencia": float(coer_nula.mean()), "referencia": "coerência com categorias sorteadas",
     "p_valor": p_coer, "conclusao": conclusao(p_coer, f"vizinhos {coer / coer_nula.mean():.1f}x mais coerentes que o acaso", "não difere do acaso")},
    {"teste": "Silhouette (cosseno)", "objeto": "missões do Item2Vec v2",
     "estatistica": SILHOUETTE, "valor_referencia": float(K), "referencia": "número de grupos escolhido", "p_valor": None,
     "conclusao": "grupos se sobrepõem (cestas de 'compra do mês' misturam corredores)" if SILHOUETTE < 0.25 else "grupos bem separados"},
    {"teste": f"Estabilidade do K-Means (ARI entre {N_ORDENS_KMEANS} ordens)", "objeto": "missões do Item2Vec v2",
     "estatistica": ARI_MEDIO, "valor_referencia": ARI_MINIMO, "referencia": "ARI médio (estatística) e mínimo (referência) entre pares de rodadas",
     "p_valor": None,
     "conclusao": "grupos estáveis" if ARI_MEDIO >= 0.8 else
                  "estabilidade moderada: o núcleo das missões se repete, as bordas mudam" if ARI_MEDIO >= 0.6 else
                  "grupos instáveis: o número de missões não é bem definido pelos dados"},
    {"teste": "McNemar exato (Item2Vec v2 x FP-Growth calibrado)", "objeto": "Hit@5 no hold-out",
     "estatistica": float(linha_fp["dif_hit_item2vec_v2_menos_modelo"]), "valor_referencia": None,
     "referencia": "diferença de Hit@5 (v2 menos FP)", "p_valor": float(linha_fp["mcnemar_p_vs_item2vec_v2"]),
     "conclusao": linha_fp["conclusao_vs_item2vec_v2"]},
    {"teste": "Bootstrap pareado (Item2Vec v2 x FP-Growth calibrado)", "objeto": "Hit@5 no hold-out",
     "estatistica": float(linha_fp["dif_ic95_inf"]), "valor_referencia": float(linha_fp["dif_ic95_sup"]),
     "referencia": "IC 95% da diferença: limite inferior (estatística) e superior (referência)", "p_valor": None,
     "conclusao": "o intervalo cruza o zero: diferença pode ser ruído" if linha_fp["dif_ic95_inf"] < 0 < linha_fp["dif_ic95_sup"]
                  else "diferença consistente"},
])
display(testes)

## 6. Insights documentados (`ml_p2_insights`)
Os achados da comparação, da evolução e da análise das missões, em frases curtas para o Looker. Os números vêm das células acima, exceto os da EDA (notebook 01) e da ablação do FP-Growth (notebook 05), indicados na coluna `fonte`.

In [ ]:
def num(x, casas=2):
    """Número no formato brasileiro: 1.234,56"""
    return f"{x:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")

def pct(x, casas=1):
    return f"{num(100 * x, casas)}%"

def pval(p):
    return "< 0,001" if p < 0.001 else num(p, 3)

v2 = comparacao[comparacao["modelo"] == "Item2Vec v2"].iloc[0]
fp = comparacao[comparacao["modelo"] == FONTES["fpgrowth_calibrado"][0]].iloc[0]
top5 = comparacao[comparacao["modelo"] == "Top-5 mais vendidos"].iloc[0]
pets = missoes.sort_values("preco_mediano", ascending=False).iloc[0]
mais_coesa = missoes.sort_values("pct_categoria_dominante", ascending=False).iloc[0]
puras = missoes[missoes["pct_categoria_dominante"] >= 0.6].sort_values("pct_categoria_dominante", ascending=False)
N_PURAS = len(puras)
mais_vendido = pd.Series([s for c in cestas_treino for s in c]).value_counts()

insights = pd.DataFrame([
    ("Generalização", "Quase todo acerto vem das cestas de teste que repetem uma cesta do treino; nas cestas novas nenhum modelo supera os mais vendidos.",
     f"Item2Vec v2: {int(v2.acertos_cestas_repetidas)}/{int(rep_mask.sum())} repetidas e {int(v2.acertos_cestas_novas)}/{int((~rep_mask).sum())} novas · "
     f"FP calibrado: {int(fp.acertos_cestas_repetidas)} e {int(fp.acertos_cestas_novas)} · mais vendidos: {int(top5.acertos_cestas_repetidas)} e {int(top5.acertos_cestas_novas)}",
     "notebook 07 (seção 2)"),
    ("Qualidade dos dados", "Uma parte grande das cestas do stream é cópia exata de outra cesta (mesmo conjunto de SKUs, pedido diferente).",
     f"{CESTAS_COPIADAS} de {num(len(cestas_treino), 0)} cestas de treino ({pct(CESTAS_COPIADAS / len(cestas_treino), 0)})",
     "notebook 07 (seção 1)"),
    ("Comparação", "Item2Vec v2 e o melhor FP-Growth empatam no acerto; o Item2Vec cobre bem mais o catálogo.",
     f"Hit@5 {pct(v2.hit_at_5, 2)} × {pct(fp.hit_at_5, 2)} (McNemar p = {num(fp.mcnemar_p_vs_item2vec_v2)}) · cobertura {pct(v2.cobertura_catalogo)} × {pct(fp.cobertura_catalogo)}",
     "notebook 07 (seção 2)"),
    ("FP-Growth", "A queda de cobertura do FP-Growth (9,2% para 1%) vem do min_support mais alto, que zera as regras de SKU, e não da divisão por período.",
     f"min_support 0,01 pede {MIN_OCORRENCIAS_V2} ocorrências; o par mais frequente aparece em {MAX_PAR} cestas",
     "notebook 07 (seção 2)"),
    ("Entradas do modelo", "Hora do pedido e período do dia não ajudam a recomendar: não se relacionam com o que vai na cesta e, ao segmentar, cortam a cobertura sem ganhar acerto.",
     "|r| ≤ 0,20 entre hora e cesta (EDA) · FP com período 4,8% / cobertura 4,2% × sem período 4,5% / 8,7%",
     "notebooks 01 e 05"),
    ("Entradas do modelo", "Treinar mais (200 épocas) aumenta o acerto, mas sozinho faz o modelo decorar; a subcategoria como palavra extra devolve o sentido aos vetores.",
     "validação: coerência 41,6% (v1) → 35,3% (só épocas) → 38,1% (com subcategoria), acerto 10,9% → 20,1%",
     "notebook 06"),
    ("Missões de compra", f"O Item2Vec organiza o catálogo em {K} missões de compra: {N_PURAS} são de um corredor só ({' · '.join(puras['categoria_raiz_dominante'])}) e {K - N_PURAS} misturam corredores, como uma 'compra do mês'.",
     f"Cramér's V {num(V)} contra {num(v_nulo.mean())} por acaso (p {pval(p_v)}) · missão mais coesa: {mais_coesa.missao} com {pct(mais_coesa.pct_categoria_dominante, 0)} de uma só categoria",
     "notebook 07 (seções 4 e 5)"),
    ("Missões de compra", "O número de missões não é bem marcado nos dados: o silhouette quase não muda entre 7 e 20 grupos, e o K-Means muda os grupos conforme a ordem dos produtos. Com o K escolhido, o núcleo das missões se repete.",
     f"silhouette médio de {num(silhuetas.silhouette_medio.min(), 3)} a {num(silhuetas.silhouette_medio.max(), 3)} entre 4 e 20 grupos · "
     f"com {K} grupos, ARI médio {num(ARI_MEDIO)} entre {N_ORDENS_KMEANS} ordens (1 = mesmos grupos)",
     "notebook 07 (seções 4 e 5)"),
    ("Missões de compra", "As missões têm perfis de preço diferentes.",
     f"Kruskal-Wallis p {pval(p_preco)} · missão mais cara: {pets.missao}, mediana R$ {num(pets.preco_mediano)} contra R$ {num(preco_mediano_geral)} no geral",
     "notebook 07 (seções 4 e 5)"),
    ("Sentido das recomendações", "Os vizinhos de cada produto caem na mesma categoria raiz muito mais do que por acaso.",
     f"coerência {pct(coer)} contra {pct(coer_nula.mean())} sorteando categorias (p {pval(p_coer)})",
     "notebook 07 (seção 5)"),
    ("Popularidade", "Nenhum produto domina as cestas, por isso recomendar os mais vendidos acerta pouco.",
     f"o SKU mais vendido está em {mais_vendido.iloc[0]} de {num(len(cestas_treino), 0)} cestas ({pct(mais_vendido.iloc[0] / len(cestas_treino))}) · Hit@5 dos mais vendidos {pct(top5.hit_at_5, 2)}",
     "notebook 07 (seção 1)"),
], columns=["tema", "insight", "evidencia", "fonte"])
insights.insert(0, "ordem", range(1, len(insights) + 1))
display(insights)

## 7. Gravação: Delta na Gold e Azure SQL Server
Cada tabela vai para os dois destinos, em overwrite: é sempre a foto atual dos resultados.
* **Delta:** `fase2_ml/gold/ml_p2_*`.
* **SQL Server:** `squad2.ml_p2_*`, pelo conector `sqlserver` do Databricks Serverless, com o mesmo cuidado da Gold da Fase 1: TRUNCATE + INSERT para manter a tabela, e recriação só se as colunas mudarem.

Listas viram texto antes de gravar, porque o SQL Server não tem tipo array.

In [ ]:
def salvar_sql(df, nome_tabela):
    """Overwrite no SQL Server; recria a tabela só se as colunas mudaram (schema drift)."""
    truncate = "true"
    try:
        colunas_sql = spark.read.format("sqlserver").options(**opcoes_sql).option("dbtable", nome_tabela).load().columns
        if {c.lower() for c in colunas_sql} != {c.lower() for c in df.columns}:
            truncate = "false"
            print(f"  [SCHEMA DRIFT] colunas de {nome_tabela} mudaram: a tabela será recriada")
    except Exception as e:
        if not any(m in str(e) for m in ("Invalid object name", "TABLE_OR_VIEW_NOT_FOUND", "does not exist")):
            raise
    (df.write.format("sqlserver").options(**opcoes_sql).option("dbtable", nome_tabela)
       .option("truncate", truncate).mode("overwrite").save())

atualizado_em = datetime.now()
tabelas = {
    "ml_p2_comparacao_modelos": comparacao,
    "ml_p2_evolucao_item2vec": evolucao,
    "ml_p2_missoes_compra": missoes,
    "ml_p2_missoes_skus": info[["sku", "grupo", "missao", "subcategoria", "categoria_raiz", "preco_lista", "freq_treino"]],
    "ml_p2_testes_estatisticos": testes,
    "ml_p2_insights": insights,
}
for nome, pdf in tabelas.items():
    pdf = pdf.copy()
    pdf["atualizado_em"] = atualizado_em
    sdf = spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None))
    gravar_delta(sdf, f"{GOLD}/{nome}")
    salvar_sql(sdf, f"squad2.{nome}")
    print(f"{nome}: {len(pdf)} linhas -> Delta e squad2.{nome}")

In [ ]:
conferencia = [{"tabela": nome,
                "linhas_delta": ler_delta(f"{GOLD}/{nome}").count(),
                "linhas_sql": spark.read.format("sqlserver").options(**opcoes_sql).option("dbtable", f"squad2.{nome}").load().count()}
               for nome in SAIDAS]
display(pd.DataFrame(conferencia))

## 8. Resultado

> **Primeira execução no Databricks (26/11/2026):** as 6 tabelas foram gravadas na Gold e no SQL Server, com as mesmas contagens nos dois destinos. A tabela única, a evolução e os testes da comparação bateram com a execução local. **O K-Means não bateu:** deu 11 grupos no Databricks e 13 no local, com os mesmos vetores.
>
> **Causa:** o K-Means depende da ordem das linhas, e o `toPandas()` do Spark não garante ordem. Testando 5 ordens dos mesmos vetores, o "melhor" K variou entre 8 e 19, porque o silhouette fica quase plano nessa faixa.
>
> **Correção (esta versão):** os vetores são ordenados por SKU, e K é escolhido pelo silhouette médio em 10 ordens diferentes, com a estabilidade (ARI) entre elas registrada como teste. Os números abaixo são da execução local desta versão, com o scikit-learn do Databricks (1.5.2). **Falta rodar de novo no Databricks** para regravar as tabelas.

### O que vai para o Looker
| Tabela (`squad2.*` e `fase2_ml/gold/*`) | Linhas | Para que serve no Looker |
|---|---|---|
| `ml_p2_comparacao_modelos` | 7 | Tabela única da comparação, com IC, cobertura, cestas repetidas × novas e o teste contra o Item2Vec v2 |
| `ml_p2_evolucao_item2vec` | 8 | Evolução v1 → v2 (validação e hold-out) |
| `ml_p2_missoes_compra` | 7 | Uma linha por missão de compra, com estatísticas de preço e popularidade |
| `ml_p2_missoes_skus` | 1.550 | Cada SKU com a missão dele (detalhe e filtros) |
| `ml_p2_testes_estatisticos` | 9 | Testes, p-valores e conclusões |
| `ml_p2_insights` | 11 | Insights em texto curto |

### Avaliação dos resultados
1. **Comparação** (igual nas duas execuções):
   * O Item2Vec v2 acerta mais que os mais vendidos, que as duas versões do FP-Growth v2 e que a própria v1 (McNemar p < 0,01 em todos os casos).
   * Contra o FP-Growth calibrado é **empate**: 22 × 20 acertos, McNemar p = 0,69, e o intervalo do bootstrap para a diferença vai de −0,6 a +2,1 pontos, cruzando o zero.
2. **O que o modelo aprendeu:** o Item2Vec organiza o catálogo em 7 missões de compra.
   * 4 são de um corredor só: Pets (97% de pet), matinais e biscoitos (82%), higiene com shampoos e fraldas (69%) e limpeza (68%).
   * As outras 3 misturam higiene com mercearia ou bebidas, como uma "compra do mês".
3. **Os testes confirmam que isso não é acaso:**
   * as missões estão associadas às categorias raiz (Cramér's V de 0,70, contra 0,08 com as categorias embaralhadas; p < 0,001);
   * os 5 vizinhos de cada SKU caem na mesma categoria raiz em 41,5% das vezes, contra 13,3% por sorteio (3,1 vezes mais; p < 0,001). Esse teste não depende do agrupamento e deu o mesmo valor nas duas execuções;
   * preço e popularidade mudam entre as missões (Kruskal-Wallis, p < 0,001). Pets é a missão mais cara, com mediana de R$ 37,04, contra R$ 19,90 no geral.
4. **Limites:**
   * o número de missões não é bem marcado nos dados: o silhouette médio fica entre 0,12 e 0,13 de 7 a 20 grupos, e é baixo (0,13), porque as missões se sobrepõem;
   * com 7 grupos, o ARI entre ordens é 0,75 (estabilidade moderada): o núcleo das missões se repete, mas os SKUs da borda trocam de grupo. Com 13 grupos ele cairia para 0,57;
   * a generalização continua sendo o ponto fraco: nas 316 cestas novas, o Item2Vec v2 acerta 4, o mesmo que os mais vendidos.

### Próximos passos
1. Rodar este notebook de novo no Databricks e conferir na última célula as contagens no SQL Server (7, 8, 7, 1.550, 9 e 11).
2. Montar a página "P2 · Recomendação no carrinho" no Looker oficial, seguindo `squad2/dashboard/GUIA_LOOKER_P2_ML.md`.